# Abalone Classification

abalone 데이터셋을 이용해 전복의 성별(Sex)을 분류하는 딥러닝 분류 문제를 수행하였다.
입력 데이터(X)와 정답 데이터(y)를 나눈 뒤, y를 원-핫 인코딩하여 딥러닝 모델로 학습하였다.

- 이번 딥러닝 분류 테스트의 정확도는 약 0.57로 측정되었다.
- confusion matrix를 보면 클래스 1은 비교적 잘 분류되었지만, 클래스 0과 클래스 2에서는 오분류가 많이 발생하였다.

In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras import layers, models

In [3]:
path = '/Users/ganghyejun/Desktop/100_College/2-1/인공지능개론/AICLASS_2-1/task/src/data/abalone.csv'
df = pd.read_csv(path, index_col=0)
df

,Sex,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Rings
id,,,,,,,,,
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.1500,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.0700,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.2100,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.1550,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.0550,7
...,...,...,...,...,...,...,...,...,...
4172,F,0.565,0.450,0.165,0.8870,0.3700,0.2390,0.2490,11
4173,M,0.590,0.440,0.135,0.9660,0.4390,0.2145,0.2605,10
4174,M,0.600,0.475,0.205,1.1760,0.5255,0.2875,0.3080,9


In [7]:
y = df['Sex']
X = df.drop('Sex', axis=1)

# 1. X를 numpy로 변환 + 스케일링
scaler = StandardScaler()
X = scaler.fit_transform(X)

# 2. train/test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0
)

### 3. y를 원-핫 인코딩
- 범주형 데이터를 라벨 인코딩으로 숫자만 부여하면, 모델이 클래스 사이에 실제로 없는 순서나 크기 관계가 있다고 잘못 인식할 수 있다.
- 이런 문제를 해결하기 위해 각 클래스를 독립적으로 표현하는 원-핫 인코딩을 사용한다.
- 원-핫 인코딩은 클래스마다 별도의 컬럼을 만들고, 해당 클래스만 1, 나머지는 0으로 표시하는 방식이다.
- 이 방식을 사용하면 예를 들어 0 < 1 < 2처럼 클래스에 크기 차이가 있는 것처럼 보이는 가짜 크기 관계를 제거할 수 있어, 딥러닝 분류 모델이 각 클래스를 더 올바르게 학습하는 데 도움이 된다.

---

> - sorted(y.unique())는 원-핫 인코딩 후 클래스 열 순서를 고정하기 위해 사용, 이를 통해 train/test의 라벨 구조를 동일하게 맞추고, 일부 클래스 열이 빠지거나 순서가 바뀌어 모델 출력과 정답 의미가 어긋나는 문제를 해결.

In [ ]:
classes = sorted(y.unique())
y_train = pd.get_dummies(y_train).reindex(columns=classes, fill_value=0).values
y_test = pd.get_dummies(y_test).reindex(columns=classes, fill_value=0).values

In [10]:
# 4. 딥러닝 분류 모델
model = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(y_train.shape[1], activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=16,
    verbose=1
)

Epoch 1/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5186 - loss: 0.9162 - val_accuracy: 0.5598 - val_loss: 0.8446
Epoch 2/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5214 - loss: 0.8854 - val_accuracy: 0.5678 - val_loss: 0.8299
Epoch 3/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 886us/step - accuracy: 0.5261 - loss: 0.8765 - val_accuracy: 0.5726 - val_loss: 0.8242
Epoch 4/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 887us/step - accuracy: 0.5449 - loss: 0.8648 - val_accuracy: 0.5646 - val_loss: 0.8179
Epoch 5/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 889us/step - accuracy: 0.5477 - loss: 0.8585 - val_accuracy: 0.5710 - val_loss: 0.8133
Epoch 6/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 905us/step - accuracy: 0.5461 - loss: 0.8610 - val_accuracy: 0.5805 - val_loss: 0.8146
Epoch 7/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 906us/step - accuracy: 0.5517 - loss: 0.8559 - val_accuracy: 0.5582 - val_loss: 0.8118
Epoch 8/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 935us/step - accuracy: 0.5493 - loss: 0.8511 - val_

In [5]:
# 5. 예측 및 평가
y_pred = model.predict(X_test)

y_test_class = np.argmax(y_test, axis=1)
y_pred_class = np.argmax(y_pred, axis=1)

print(classification_report(y_test_class, y_pred_class))
print(confusion_matrix(y_test_class, y_pred_class))

33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
              precision    recall  f1-score   support

           0       0.48      0.48      0.48       316
           1       0.71      0.78      0.75       359
           2       0.50      0.45      0.47       370

    accuracy                           0.57      1045
   macro avg       0.56      0.57      0.57      1045
weighted avg       0.57      0.57      0.57      1045

[[153  43 120]
 [ 35 281  43]
 [133  71 166]]
